# Phân tích trường `ProcessName` bị thiếu — ngày 16

Notebook kiểm tra đồng thời dữ liệu **Event 4624** (Successful Logon) và **Event 4625** (Failed Logon).

Khác với `DomainName` (chỉ thiếu 37 dòng), `ProcessName` thiếu hơn **16.3 triệu dòng** (chiếm ~92% ở 4624 và ~49% ở 4625).
Do đó, notebook sử dụng kỹ thuật **streaming/batch processing qua PyArrow** để giới hạn RAM tiêu thụ (< 500MB), phân rã bản chất thiếu theo `LogonType`, khảo sát phân bố các tiến trình khi có mặt, và trích xuất mẫu kiểm chứng an toàn mà không làm sập bộ nhớ.

In [1]:
from collections import Counter
from pathlib import Path
import sys
from typing import Any, Dict, List, Optional, Tuple

# Đảm bảo hiển thị an toàn UTF-8 trên console Windows
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")

import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
from IPython.display import display

# Cấu hình tham số - dễ dàng mở rộng và tái sử dụng cho các ngày hoặc cột khác
DAY_TAG = "day-16"
TARGET_COLUMN = "ProcessName"
BATCH_SIZE = 250_000
SAMPLE_LIMIT = 20

PARQUET_RELATIVE_PATHS = {
    "Event 4624": Path(f"data/interim/event_4624/event_4624_{DAY_TAG}.parquet"),
    "Event 4625": Path(f"data/interim/event_4625/event_4625_{DAY_TAG}.parquet"),
}


def find_project_file(relative_path: Path, start_directory: Optional[Path] = None) -> Path:
    """Tìm file từ thư mục hiện tại hoặc một trong các thư mục cha."""
    start_directory = (start_directory or Path.cwd()).resolve()

    for directory in (start_directory, *start_directory.parents):
        candidate = directory / relative_path
        if candidate.is_file():
            return candidate

    raise FileNotFoundError(
        f"Không tìm thấy '{relative_path}' từ '{start_directory}' hoặc các thư mục cha."
    )


parquet_paths = {
    event_name: find_project_file(relative_path)
    for event_name, relative_path in PARQUET_RELATIVE_PATHS.items()
}

for event_name, parquet_path in parquet_paths.items():
    print(f"{event_name}: {parquet_path}")


Event 4624: D:\Github Repo\ueba-benchmark\data\interim\event_4624\event_4624_day-16.parquet
Event 4625: D:\Github Repo\ueba-benchmark\data\interim\event_4625\event_4625_day-16.parquet


## 1. Hàm tiện ích kiểm tra giá trị thiếu và tính toán thống kê

Một giá trị được xem là thiếu nếu là `null`, chuỗi rỗng `""`, hoặc chỉ chứa khoảng trắng.
Các hàm xử lý dưới đây dùng trực tiếp `pyarrow.compute` theo batch để streaming, không nạp toàn bộ dữ liệu vào Pandas DataFrame.

In [2]:
def missing_text_mask(values: pa.Array) -> pa.Array:
    """Trả về mask boolean cho null, chuỗi rỗng và chuỗi chỉ có khoảng trắng."""
    values_without_null = pc.fill_null(values, "")
    return pc.equal(pc.utf8_trim_whitespace(values_without_null), "")


def compute_column_missing_stats(
    file_path: Path,
    column: str,
    batch_size: int = BATCH_SIZE,
) -> Dict[str, Any]:
    """Tính tổng số dòng, số dòng thiếu và số dòng hợp lệ theo streaming batch."""
    parquet_file = pq.ParquetFile(file_path)
    if column not in parquet_file.schema_arrow.names:
        raise KeyError(f"Không có cột '{column}' trong '{file_path}'.")

    total_rows = 0
    missing_rows = 0

    for batch in parquet_file.iter_batches(columns=[column], batch_size=batch_size):
        col_data = batch.column(0)
        total_rows += len(col_data)
        mask = missing_text_mask(col_data)
        missing_rows += pc.sum(mask.cast(pa.int64())).as_py()

    valid_rows = total_rows - missing_rows
    missing_rate = (missing_rows / total_rows * 100) if total_rows > 0 else 0.0

    return {
        "TotalRows": total_rows,
        "MissingRows": missing_rows,
        "ValidRows": valid_rows,
        "MissingRatePct": round(missing_rate, 2),
    }


## 2. Thống kê tổng quan tỷ lệ thiếu `ProcessName`

So sánh tổng thể giữa Event 4624 (đăng nhập thành công) và Event 4625 (đăng nhập thất bại).

In [3]:
overview_records = []
for event_name, path in parquet_paths.items():
    stats = compute_column_missing_stats(path, TARGET_COLUMN)
    stats["Event"] = event_name
    overview_records.append(stats)

df_overview = pd.DataFrame(overview_records)[
    ["Event", "TotalRows", "MissingRows", "ValidRows", "MissingRatePct"]
]
display(df_overview)


,Event,TotalRows,MissingRows,ValidRows,MissingRatePct
0,Event 4624,17553517,16213560,1339957,92.37
1,Event 4625,359897,176924,182973,49.16


## 3. Phân rã tỷ lệ thiếu theo `LogonType`

Windows Event 4624/4625 ghi nhận nhiều kiểu đăng nhập khác nhau.
Bảng dưới đây thống kê số lượng và tỷ lệ thiếu `ProcessName` phân bổ theo từng `LogonType` và `LogonTypeDescription`.

In [4]:
def compute_missing_by_logon_type(
    file_path: Path,
    target_column: str = TARGET_COLUMN,
    batch_size: int = BATCH_SIZE,
) -> pd.DataFrame:
    """Thống kê số lượng missing và valid theo từng cặp (LogonType, LogonTypeDescription)."""
    parquet_file = pq.ParquetFile(file_path)
    required_cols = ["LogonType", "LogonTypeDescription", target_column]
    for col in required_cols:
        if col not in parquet_file.schema_arrow.names:
            raise KeyError(f"Thiếu cột '{col}' trong '{file_path}'.")

    counts: Dict[Tuple[Any, Any], Dict[str, int]] = {}

    for batch in parquet_file.iter_batches(columns=required_cols, batch_size=batch_size):
        df_batch = batch.to_pandas()
        is_missing = df_batch[target_column].isna() | df_batch[target_column].astype(str).str.strip().eq("")

        grouped = df_batch.groupby(["LogonType", "LogonTypeDescription"], dropna=False).size()
        missing_grouped = df_batch[is_missing].groupby(["LogonType", "LogonTypeDescription"], dropna=False).size()

        for key, total_cnt in grouped.items():
            if key not in counts:
                counts[key] = {"Total": 0, "Missing": 0}
            counts[key]["Total"] += int(total_cnt)
            counts[key]["Missing"] += int(missing_grouped.get(key, 0))

    rows = []
    for (ltype, ldesc), val in counts.items():
        total = val["Total"]
        missing = val["Missing"]
        valid = total - missing
        rate = round(missing / total * 100, 2) if total > 0 else 0.0
        rows.append({
            "LogonType": ltype,
            "LogonTypeDescription": ldesc,
            "Total": total,
            "Missing": missing,
            "Valid": valid,
            "MissingRatePct": rate,
        })

    df_res = pd.DataFrame(rows).sort_values(by="Total", ascending=False).reset_index(drop=True)
    return df_res


for event_name, path in parquet_paths.items():
    print(f"\n{'=' * 80}\n{event_name}: Phân rã missing theo LogonType\n{'=' * 80}")
    df_ltype = compute_missing_by_logon_type(path)
    display(df_ltype)



Event 4624: Phân rã missing theo LogonType


,LogonType,LogonTypeDescription,Total,Missing,Valid,MissingRatePct
0,3,Network,16794417,16207774,586643,96.51
1,5,Service,373734,0,373734,0.00
2,2,Interactive,219613,0,219613,0.00
3,7,Unlock,78760,0,78760,0.00
4,8,NetworkClearText,62286,0,62286,0.00
5,9,NewCredentials,13451,0,13451,0.00
6,0,Local System,5786,5786,0,100.00
7,4,Batch,2321,0,2321,0.00
8,10,RemoteInteractive,2230,0,2230,0.00
9,11,CachedInteractive,916,0,916,0.00



Event 4625: Phân rã missing theo LogonType


,LogonType,LogonTypeDescription,Total,Missing,Valid,MissingRatePct
0,3,Network,334377,176924,157453,52.91
1,2,Interactive,18872,0,18872,0.00
2,7,Unlock,3624,0,3624,0.00
3,4,Batch,1906,0,1906,0.00
4,11,CachedInteractive,588,0,588,0.00
5,10,RemoteInteractive,267,0,267,0.00
6,5,Service,146,0,146,0.00
7,8,NetworkClearText,117,0,117,0.00


## 4. Phân tích các tiến trình hợp lệ (`Valid ProcessName`)

Khảo sát các tiến trình xuất hiện nhiều nhất khi `ProcessName` không bị thiếu.

In [5]:
def get_top_process_names(
    file_path: Path,
    target_column: str = TARGET_COLUMN,
    top_n: int = 15,
    batch_size: int = BATCH_SIZE,
) -> Tuple[pd.DataFrame, int, int]:
    """Đếm tần suất xuất hiện của các ProcessName hợp lệ bằng Counter streaming."""
    parquet_file = pq.ParquetFile(file_path)
    counter: Counter = Counter()
    total_valid = 0

    for batch in parquet_file.iter_batches(columns=[target_column], batch_size=batch_size):
        col_data = batch.column(0)
        mask = pc.invert(missing_text_mask(col_data))
        valid_items = pc.filter(col_data, mask).to_pylist()
        counter.update(valid_items)
        total_valid += len(valid_items)

    most_common = counter.most_common(top_n)
    rows = []
    for proc_name, count in most_common:
        rows.append({
            "ProcessName": proc_name,
            "Count": count,
            "PercentageOfValid": round(count / total_valid * 100, 2) if total_valid > 0 else 0.0,
        })

    df_top = pd.DataFrame(rows)
    return df_top, len(counter), total_valid


for event_name, path in parquet_paths.items():
    df_top_procs, num_unique, total_valid = get_top_process_names(path)
    print(f"\n{event_name}: Có {num_unique:,} tiến trình duy nhất (trên tổng {total_valid:,} dòng có ProcessName)")
    display(df_top_procs)



Event 4624: Có 65 tiến trình duy nhất (trên tổng 1,339,957 dòng có ProcessName)


,ProcessName,Count,PercentageOfValid
0,services.exe,373501,27.87
1,Proc415255.exe,205817,15.36
2,Proc813133.exe,194351,14.50
3,Proc811977.exe,182480,13.62
4,Proc414897.exe,129493,9.66
5,lsass.exe,92000,6.87
6,winlogon.exe,58696,4.38
7,Proc423620.exe,41592,3.10
8,Proc570610.exe,10913,0.81
9,Proc252919.exe,10567,0.79



Event 4625: Có 38 tiến trình duy nhất (trên tổng 182,973 dòng có ProcessName)


,ProcessName,Count,PercentageOfValid
0,Proc275807.exe,156505,85.53
1,Proc813133.exe,17085,9.34
2,winlogon.exe,5274,2.88
3,svchost.exe,1949,1.07
4,Proc858929.exe,745,0.41
5,Proc641217.exe,263,0.14
6,lsass.exe,161,0.09
7,Proc089893.exe,158,0.09
8,Proc811977.exe,147,0.08
9,Proc013220.exe,137,0.07


## 5. Mối tương quan thiếu giữa `ProcessName` và `ParentProcessName`

Kiểm tra xem khi `ProcessName` thiếu thì `ParentProcessName` có thiếu đồng thời hay không.

In [ ]:
def check_process_pair_completeness(
    file_path: Path,
    batch_size: int = BATCH_SIZE,
) -> pd.DataFrame:
    """Khảo sát tính đồng thời giữa ProcessName và ParentProcessName."""
    parquet_file = pq.ParquetFile(file_path)
    pair_counts: Counter = Counter()

    for batch in parquet_file.iter_batches(
        columns=["ProcessName", "ParentProcessName"],
        batch_size=batch_size,
    ):
        p_mask = missing_text_mask(batch.column(0)).to_pylist()
        pp_mask = missing_text_mask(batch.column(1)).to_pylist()
        for p_miss, pp_miss in zip(p_mask, pp_mask):
            pair_counts[(p_miss, pp_miss)] += 1

    total = sum(pair_counts.values())
    rows = [
        {
            "ProcessName_Missing": p_m,
            "ParentProcessName_Missing": pp_m,
            "Count": cnt,
            "Percentage": round(cnt / total * 100, 2) if total > 0 else 0.0,
        }
        for (p_m, pp_m), cnt in pair_counts.items()
    ]
    return pd.DataFrame(rows).sort_values(by="Count", ascending=False).reset_index(drop=True)


for event_name, path in parquet_paths.items():
    print(f"\n{'=' * 80}\n{event_name}: Ma trận tương quan thiếu ProcessName vs ParentProcessName\n{'=' * 80}")
    df_pair = check_process_pair_completeness(path)
    display(df_pair)


## 6. Trích xuất mẫu kiểm chứng an toàn (Safe Sample Preview)

Trích xuất có giới hạn tối đa `SAMPLE_LIMIT` dòng đầu tiên bị thiếu `ProcessName` để khảo sát bối cảnh các trường liên quan (Source, LogHost, UserName, AuthenticationPackage...) mà không nạp 16 triệu dòng vào RAM.

In [6]:
def sample_missing_rows(
    file_path: Path,
    target_column: str = TARGET_COLUMN,
    sample_limit: int = SAMPLE_LIMIT,
    batch_size: int = BATCH_SIZE,
) -> pd.DataFrame:
    """Trích xuất an toàn tối đa sample_limit dòng thiếu đầu tiên."""
    parquet_file = pq.ParquetFile(file_path)
    col_names = parquet_file.schema_arrow.names
    target_idx = col_names.index(target_column)

    collected_batches: List[pa.RecordBatch] = []
    collected_count = 0

    for batch in parquet_file.iter_batches(batch_size=batch_size):
        mask = missing_text_mask(batch.column(target_idx))
        if pc.any(mask).as_py():
            filtered = batch.filter(mask)
            needed = sample_limit - collected_count
            if len(filtered) > needed:
                filtered = filtered.slice(0, needed)
            collected_batches.append(filtered)
            collected_count += len(filtered)
            if collected_count >= sample_limit:
                break

    if not collected_batches:
        return pd.DataFrame(columns=col_names)

    sampled_table = pa.Table.from_batches(collected_batches)
    return sampled_table.to_pandas()


display_cols = [
    "Time", "EventID", "LogHost", "LogonType", "LogonTypeDescription",
    "UserName", "DomainName", "Source", "AuthenticationPackage", "ProcessName", "ParentProcessName"
]

for event_name, path in parquet_paths.items():
    print(f"\n{'=' * 80}\n{event_name}: Mẫu {SAMPLE_LIMIT} dòng đầu tiên bị thiếu {TARGET_COLUMN}\n{'=' * 80}")
    df_sample = sample_missing_rows(path, sample_limit=SAMPLE_LIMIT)
    display(df_sample[display_cols])



Event 4624: Mẫu 20 dòng đầu tiên bị thiếu ProcessName


,Time,EventID,LogHost,LogonType,LogonTypeDescription,UserName,DomainName,Source,AuthenticationPackage,ProcessName,ParentProcessName
0,1296000,4624,Comp284280,3,Network,Comp284280$,Domain001,NaN,Kerberos,NaN,NaN
1,1296000,4624,Comp505809,3,Network,Comp505809$,Domain001,NaN,Kerberos,NaN,NaN
2,1296000,4624,ActiveDirectory,3,Network,Comp883506$,Domain001,Comp883506,Kerberos,NaN,NaN
3,1296000,4624,ActiveDirectory,3,Network,Anonymous,nt authority,Comp275037,NTLM,NaN,NaN
4,1296000,4624,ActiveDirectory,3,Network,Anonymous,nt authority,Comp275037,NTLM,NaN,NaN
5,1296000,4624,ActiveDirectory,3,Network,Comp275037$,Domain001,Comp275037,Kerberos,NaN,NaN
6,1296000,4624,Comp186428,3,Network,User643724,Domain001,NaN,Kerberos,NaN,NaN
7,1296000,4624,Comp186428,3,Network,User643724,Domain001,NaN,Kerberos,NaN,NaN
8,1296000,4624,Comp186428,3,Network,User643724,Domain001,NaN,Kerberos,NaN,NaN
9,1296000,4624,EnterpriseAppServer,3,Network,EnterpriseAppServer$,Domain001,EnterpriseAppServer,Kerberos,NaN,NaN



Event 4625: Mẫu 20 dòng đầu tiên bị thiếu ProcessName


,Time,EventID,LogHost,LogonType,LogonTypeDescription,UserName,DomainName,Source,AuthenticationPackage,ProcessName,ParentProcessName
0,1296000,4625,Comp995183,3,Network,User031784,Domain002,Comp070206,NTLM,NaN,NaN
1,1296002,4625,Comp995183,3,Network,User031784,Domain002,Comp070206,NTLM,NaN,NaN
2,1296004,4625,ActiveDirectory,3,Network,Comp847416$,Domain001,Comp847416,NTLM,NaN,NaN
3,1296004,4625,Comp995183,3,Network,User031784,Domain002,Comp070206,NTLM,NaN,NaN
4,1296006,4625,Comp995183,3,Network,User031784,Domain002,Comp070206,NTLM,NaN,NaN
5,1296007,4625,ActiveDirectory,3,Network,Comp847416$,Domain001,Comp847416,NTLM,NaN,NaN
6,1296009,4625,Comp995183,3,Network,User031784,Domain002,Comp070206,NTLM,NaN,NaN
7,1296011,4625,Comp995183,3,Network,User031784,Domain002,Comp070206,NTLM,NaN,NaN
8,1296012,4625,ActiveDirectory,3,Network,Comp847416$,Domain001,Comp847416,NTLM,NaN,NaN
9,1296015,4625,Comp180216,3,Network,Scanner,Domain001,NaN,NTLM,NaN,NaN


## 7. Kết luận và Khuyến nghị cho UEBA Benchmark

1. **Bản chất của `ProcessName` thiếu:**
   - Tỷ lệ thiếu cao (~92% ở 4624 và ~49% ở 4625) không phải là do lỗi chất lượng dữ liệu ngẫu nhiên (data corruption), mà là đặc tính cơ chế Windows Event Logging đối với **Network Logon (LogonType 3)**.
   - Đối với các kiểu logon tương tác hoặc dịch vụ (`Interactive` - 2, `Service` - 5, `Unlock` - 7, `RemoteInteractive` - 10), `ProcessName` luôn có mặt đầy đủ (tỷ lệ thiếu 0%).

2. **Đề xuất Tiền xử lý (Preprocessing):**
   - Tuyệt đối **không** dùng imputation trung vị hay mode để lấp giá trị thiếu cho `ProcessName`.
   - Nên coi trạng thái thiếu là một giá trị định danh nghiệp vụ riêng, ví dụ `"<NETWORK_NO_PROCESS>"` hoặc kết hợp với feature nhị phân `HasProcessName`.

3. **Đề xuất Kỹ thuật đặc trưng (Feature Engineering):**
   - Với logon cục bộ/service: `ProcessName` là tín hiệu quan trọng để phát hiện tiến trình độc hại / bất thường (`Proc*.exe` lạ, tiến trình ngoài baseline).
   - Với logon qua mạng: trọng tâm phân tích chuyển dịch sang bộ ba `(Source, LogHost, AuthenticationPackage)`.